In [ ]:
import os
import json
import copy
import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
from tqdm import tqdm

data_root = "."
train_dir = os.path.join(data_root, "splitted", "train")
test_dir = os.path.join(data_root, "splitted", "test")
annot_csv = os.path.join(data_root, "train_annotations.csv")
labels_json = os.path.join(data_root, "labels.json")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

img_size = 256
batch_size = 12
epochs = 15
lr = 1e-3
n_folds = 5

In [ ]:
with open(labels_json, "r", encoding="utf-8") as f:
    label_map = json.load(f)

name2idx = {name: int(idx) for idx, name in label_map.items()}
num_classes = len(label_map)
print(f"Классов: {num_classes}")

df = pd.read_csv(annot_csv)
df = df[df["split"] == "train"].reset_index(drop=True)

def build_path(row):
    p = row["image_path"].replace("\\", "/")
    return os.path.join(train_dir, p.split("/", 1)[1])

df["full_path"] = df.apply(build_path, axis=1)
df["label_idx"] = df["class"].map(name2idx)
df = df.dropna(subset=["label_idx"]).reset_index(drop=True)
df["label_idx"] = df["label_idx"].astype(int)

print(f"Всего картинок: {len(df)}")
print(df.head())

In [ ]:
mean = [0.485, 0.456, 0.406]
std = [0.229, 0.224, 0.225]

train_tfms = transforms.Compose([
    transforms.Resize((img_size + 32, img_size + 32)),
    transforms.RandomResizedCrop(img_size, scale=(0.6, 1.0)), 
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])

val_tfms = transforms.Compose([
    transforms.Resize((img_size, img_size)),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])

class OnePieceDataset(Dataset):
    def __init__(self, df, tfms):
        self.paths = df["full_path"].tolist()
        self.labels = df["label_idx"].tolist()
        self.tfms = tfms
        
    def __len__(self):
        return len(self.paths)
        
    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert("RGBA")
        background = Image.new("RGB", img.size, (255, 255, 255))
        background.paste(img, mask=img.split()[3]) 
        return self.tfms(background), self.labels[idx]

In [ ]:
print("\nОбучение EfficientNet-B0...")
skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=42)
models_list = []

for fold, (train_idx, val_idx) in enumerate(skf.split(df, df["label_idx"])):
    print(f"\nФолд {fold+1}/{n_folds}")
    train_df = df.iloc[train_idx].reset_index(drop=True)
    val_df = df.iloc[val_idx].reset_index(drop=True)
    
    train_loader = DataLoader(OnePieceDataset(train_df, train_tfms), batch_size=batch_size, shuffle=True, num_workers=0)
    val_loader = DataLoader(OnePieceDataset(val_df, val_tfms), batch_size=batch_size, shuffle=False, num_workers=0)

    model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
    model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)
    model = model.to(device)
    
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    
    best_f1 = 0.0
    best_weights = copy.deepcopy(model.state_dict())
    
    for epoch in range(epochs):
        model.train()
        loop = tqdm(train_loader, desc=f"Epoch {epoch+1}", leave=False)
        for imgs, labels in loop:
            imgs, labels = imgs.to(device), labels.to(device)
            optimizer.zero_grad()
            loss = criterion(model(imgs), labels)
            loss.backward()
            optimizer.step()
            loop.set_postfix(loss=loss.item())
        scheduler.step()
        
        model.eval()
        all_preds, all_targets = [], []
        with torch.no_grad():
            for imgs, labels in val_loader:
                imgs, labels = imgs.to(device), labels.to(device)
                all_preds.extend(model(imgs).argmax(1).cpu().numpy())
                all_targets.extend(labels.cpu().numpy())
        val_f1 = f1_score(all_targets, all_preds, average="macro")
        
        if val_f1 > best_f1:
            best_f1 = val_f1
            best_weights = copy.deepcopy(model.state_dict())
            
    print(f"Лучший F1 на фолде: {best_f1:.4f}")
    models_list.append(best_weights)
    
    del model, train_loader, val_loader
    torch.cuda.empty_cache() if torch.cuda.is_available() else None

In [ ]:
print("\nГотовим тестовые данные...")
sub = pd.read_csv("submission_efficientnet_5fold.csv")
test_ids = sub["id"].tolist()
test_files = os.listdir(test_dir)
stem2file = {fn.rsplit(".", 1)[0]: os.path.join(test_dir, fn) for fn in test_files}

class TestDatasetTTA(Dataset):
    def __init__(self, ids, stem2file, tfms):
        self.ids = ids
        self.stem2file = stem2file
        self.tfms = tfms
        
    def __len__(self):
        return len(self.ids)
        
    def __getitem__(self, idx):
        img_id = self.ids[idx]
        img = Image.open(self.stem2file[img_id]).convert("RGBA")
        background = Image.new("RGB", img.size, (255, 255, 255))
        background.paste(img, mask=img.split()[3])
        
        img1 = self.tfms(background)
        img2 = self.tfms(transforms.functional.hflip(background))
        img3 = self.tfms(transforms.functional.rotate(background, 10))
        img4 = self.tfms(transforms.functional.adjust_contrast(background, 1.2))
        
        bg5 = transforms.functional.resize(background, (350, 350))
        bg5 = transforms.functional.center_crop(bg5, (img_size, img_size))
        img5 = self.tfms(bg5)
        
        return img1, img2, img3, img4, img5

test_loader = DataLoader(TestDatasetTTA(test_ids, stem2file, val_tfms), batch_size=batch_size, shuffle=False)

In [ ]:
final_probs = torch.zeros((len(test_ids), num_classes))

for i, weights in enumerate(models_list):
    print(f"Модель {i+1}/{n_folds}")
    model = models.efficientnet_b0()
    model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)
    model.load_state_dict(weights)
    model = model.to(device)
    model.eval()
    
    model_probs = []
    with torch.no_grad():
        for img1, img2, img3, img4, img5 in tqdm(test_loader, desc=f"Модель {i+1}", leave=False):
            p1 = torch.softmax(model(img1.to(device)), dim=1)
            p2 = torch.softmax(model(img2.to(device)), dim=1)
            p3 = torch.softmax(model(img3.to(device)), dim=1)
            p4 = torch.softmax(model(img4.to(device)), dim=1)
            p5 = torch.softmax(model(img5.to(device)), dim=1)
            model_probs.append(((p1 + p2 + p3 + p4 + p5) / 5.0).cpu())
            
    final_probs += torch.cat(model_probs, dim=0)
    
    del model
    torch.cuda.empty_cache() if torch.cuda.is_available() else None

final_probs /= n_folds
final_preds = final_probs.argmax(dim=1).numpy()

sub["label"] = final_preds
sub.to_csv("submission_efficientnet_256px.csv", index=False)
print("\nСабмит сохранён в submission_efficientnet_256px.csv")
print(sub.head())